# Convert the data from db to parquet file format

In [8]:
import duckdb
from pathlib import Path
import time

# ----------------------------
# CONFIG
# ----------------------------
db_file = Path('../data/landmark_database.db').resolve()  # path to your SQLite DB
table_name = 'landmarks'                                   # table you want to export
parquet_file = Path('../data/landmarks.parquet').resolve()        # output Parquet file
columns_to_export = 'id, patient_name, frame, movement_type, side, timestamp_ms, landmark_id, x_norm, y_norm, z_norm, visibility, x_px, y_px, dataset, gait_pattern, add_pattern_info, title, fps, width, height, gait_markers, file_path'                                    # list columns, or '*' for all
compression = 'ZSTD'                                       # 'SNAPPY' is faster but bigger, 'ZSTD' compresses better

# ----------------------------
# CONNECT TO DUCKDB
# ----------------------------
con = duckdb.connect()
con.execute("INSTALL sqlite; LOAD sqlite;")

# ----------------------------
# GET ROW COUNT (for progress estimate)
# ----------------------------
count_query = f"""
SELECT COUNT(*) AS total_rows
FROM sqlite_scan('{db_file}', '{table_name}')
"""
total_rows = con.execute(count_query).fetchone()[0]
print(f"Total rows to export: {total_rows:,}")

# ----------------------------
# EXPORT TO PARQUET
# ----------------------------
start_time = time.time()

export_query = f"""
COPY (
    SELECT {columns_to_export}
    FROM sqlite_scan('{db_file}', '{table_name}')
)
TO '{parquet_file}'
(FORMAT PARQUET, COMPRESSION {compression});
"""

print("Starting export to Parquet...")
con.execute(export_query)

end_time = time.time()
elapsed = end_time - start_time
print(f"Export finished in {elapsed/60:.2f} minutes")
print(f"Parquet saved to: {parquet_file}")


Total rows to export: 21,906,020
Starting export to Parquet...
Export finished in 0.44 minutes
Parquet saved to: /Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/data/landmarks.parquet


### Compare the row and columns from the parquet and .db database file

In [9]:
# check parquet file
import polars as pl

# Load lazily
lf_parquet = pl.scan_parquet("../data/landmarks.parquet")

# 1️⃣ Number of rows
parquet_rows = lf_parquet.select(pl.len()).collect()[0, 0]

# 2️⃣ Number of columns
parquet_columns = len(lf_parquet.columns)

# 3️⃣ Column names
parquet_colnames = lf_parquet.columns

print(f"Parquet rows: {parquet_rows}")
print(f"Parquet columns: {parquet_columns}")
print(f"Column names: {parquet_colnames}")


Parquet rows: 21906020
Parquet columns: 22
Column names: ['id', 'patient_name', 'frame', 'movement_type', 'side', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm', 'z_norm', 'visibility', 'x_px', 'y_px', 'dataset', 'gait_pattern', 'add_pattern_info', 'title', 'fps', 'width', 'height', 'gait_markers', 'file_path']


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2320/2942563969.py:11: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  parquet_columns = len(lf_parquet.columns)
/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2320/2942563969.py:14: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  parquet_colnames = lf_parquet.columns


In [10]:
#check .db vile
import sqlite3

# Connect to your SQLite DB
conn = sqlite3.connect("../data/landmark_database.db")
cur = conn.cursor()

# 1️⃣ Pick your table name
table_name = "landmarks"

# 2️⃣ Number of rows
cur.execute(f"SELECT COUNT(*) FROM {table_name}")
db_rows = cur.fetchone()[0]

# 3️⃣ Column names
cur.execute(f"PRAGMA table_info({table_name})")
db_columns = [col[1] for col in cur.fetchall()]
db_column_count = len(db_columns)

print(f"SQLite rows: {db_rows}")
print(f"SQLite columns: {db_column_count}")
print(f"Column names: {db_columns}")

conn.close()


SQLite rows: 21906020
SQLite columns: 36
Column names: ['id', 'patient_name', 'frame', 'movement_type', 'jacket_status', 'side', 'model_name', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm', 'z_norm', 'visibility', 'x_px', 'y_px', 'source_file', 'file_order', 'file_path', 'start_frame', 'end_frame', 'url', 'gait_event', 'dataset', 'gait_pattern', 'add_pattern_info', 'title', 'uploader', 'fps', 'start_time', 'end_time', 'duration', 'checksum', 'width', 'height', 'gait_markers', 'created_at']


In [11]:
# get results of size check
print(f"Row difference: {db_rows - parquet_rows}")
print(f"Column difference: {db_column_count - parquet_columns}")
missing_cols = set(db_columns) - set(parquet_colnames)
print(f"Columns in DB but missing in Parquet: {missing_cols}")


Row difference: 0
Column difference: 14
Columns in DB but missing in Parquet: {'jacket_status', 'model_name', 'created_at', 'end_frame', 'start_time', 'end_time', 'source_file', 'checksum', 'duration', 'gait_event', 'url', 'start_frame', 'uploader', 'file_order'}


# Load the data lazyly

In [79]:
import polars as pl

lf = pl.scan_parquet("../data/landmarks.parquet")


In [80]:
# 2️⃣ Extract filename (works for both / and \ paths) before ".csv"
lf = lf.with_columns([
    pl.col("file_path")
      .str.extract(r"([^/\\]+)\.csv$", 1)  # capture last part after / or \ and before .csv
      .alias("video_id")
])

In [81]:
# 3️⃣ Preview first 10 rows safely
# Assume lf is your LazyFrame
df_preview = lf.limit(10).collect()  # limit first 10 rows and collect into eager DataFrame
print(df_preview)

shape: (10, 23)
┌─────┬──────────────┬───────┬──────────────┬───┬────────┬─────────────┬─────────────┬─────────────┐
│ id  ┆ patient_name ┆ frame ┆ movement_typ ┆ … ┆ height ┆ gait_marker ┆ file_path   ┆ video_id    │
│ --- ┆ ---          ┆ ---   ┆ e            ┆   ┆ ---    ┆ s           ┆ ---         ┆ ---         │
│ i64 ┆ str          ┆ i64   ┆ ---          ┆   ┆ i64    ┆ ---         ┆ str         ┆ str         │
│     ┆              ┆       ┆ str          ┆   ┆        ┆ str         ┆             ┆             │
╞═════╪══════════════╪═══════╪══════════════╪═══╪════════╪═════════════╪═════════════╪═════════════╡
│ 1   ┆ PA000        ┆ 0     ┆ Fast         ┆ … ┆ null   ┆ null        ┆ C:\User_V\2 ┆ semantic_se │
│     ┆              ┆       ┆ Movement     ┆   ┆        ┆             ┆ _Github\GAI ┆ gmentation_ │
│     ┆              ┆       ┆              ┆   ┆        ┆             ┆ Ty-Capst…   ┆ PA000_FG…   │
│ 2   ┆ PA000        ┆ 1     ┆ Fast         ┆ … ┆ null   ┆ null        ┆ C:

### Filter the rows out that contain empty landmarkers a the beginning and end of the time series of the snippet

In [ ]:
# 1️⃣ Compute frame-level validity
frame_validity = (
    lf
    .group_by(["video_id", "frame"])
    .agg(
        (
            pl.col("x_norm").is_not_null() &
            pl.col("y_norm").is_not_null() &
            pl.col("z_norm").is_not_null()
        ).all().alias("frame_valid")
    )
)

# 2️⃣ Find first and last valid frame per video
bounds = (
    frame_validity
    .filter(pl.col("frame_valid"))
    .group_by("video_id")
    .agg([
        pl.min("frame").alias("first_valid_frame"),
        pl.max("frame").alias("last_valid_frame"),
    ])
)

#3️⃣ Trim using frame numbers (not row_idx)
lf_clean = (
    lf
    .join(bounds, on="video_id", how="inner")
    .filter(
        (pl.col("frame") >= pl.col("first_valid_frame")) &
        (pl.col("frame") <= pl.col("last_valid_frame"))
    )
    .drop(["first_valid_frame", "last_valid_frame"])
)

#4️⃣ Final sanity check (this SHOULD now be empty)
check = (
    lf_clean
    .group_by(["video_id", "frame"])
    .agg(
        (
            pl.col("x_norm").is_not_null() &
            pl.col("y_norm").is_not_null() &
            pl.col("z_norm").is_not_null()
        ).all().alias("frame_valid")
    )
    .sort(["video_id", "frame"])   # 🔴 THIS WAS MISSING
    .group_by("video_id")
    .agg([
        pl.first("frame_valid").alias("first_frame_valid"),
        pl.last("frame_valid").alias("last_frame_valid"),
    ])
    .collect()
)

print(check.filter(~pl.col("first_frame_valid") | ~pl.col("last_frame_valid")))

# Save the cleaned dataset
lf_clean.sink_parquet("../data/landmarks_cleaned.parquet")


shape: (0, 3)
┌──────────┬───────────────────┬──────────────────┐
│ video_id ┆ first_frame_valid ┆ last_frame_valid │
│ ---      ┆ ---               ┆ ---              │
│ str      ┆ bool              ┆ bool             │
╞══════════╪═══════════════════╪══════════════════╡
└──────────┴───────────────────┴──────────────────┘


### Sorted according to patients

In [ ]:
# 2️⃣ Mark invalid rows
lf = lf.with_columns(
    (
        pl.col("x_norm").is_null() |
        pl.col("y_norm").is_null() |
        pl.col("z_norm").is_null()
    ).alias("row_invalid")
)


#2️⃣ 🔴 NEW: promote row invalidity → frame invalidity
lf = lf.with_columns(
    pl.any("row_invalid")
      .over(["patient_name", "frame"])
      .alias("frame_invalid")
)


# 3️⃣ Sort
lf = lf.sort(["patient_name", "frame"])

# 4️⃣ Detect sequences
lf = lf.with_columns(
    (
        (pl.col("frame") - pl.col("frame").shift(1).over("patient_name") != 1)
        .fill_null(True)
        .cast(pl.UInt32)
    )
    .cum_sum()
    .over("patient_name")
    .alias("sequence_id")
)


# 5️⃣ Assign a row index per (patient_name, sequence)
lf = lf.with_columns(
    pl.int_range(0, pl.len())
      .over(["patient_name", "sequence_id"])
      .alias("row_idx")
)



# 6️⃣ Find the first and last valid frame row per sequence
bounds = (
    lf.filter(~pl.col("frame_invalid"))
      .group_by(["patient_name", "sequence_id"])
      .agg([
          pl.min("row_idx").alias("first_valid_idx"),
          pl.max("row_idx").alias("last_valid_idx"),
      ])
)

# Trim only boundary invalid frames 
lf_clean = (
    lf.join(bounds, on=["patient_name", "sequence_id"], how="inner")
      .filter(
          (pl.col("row_idx") >= pl.col("first_valid_idx")) &
          (pl.col("row_idx") <= pl.col("last_valid_idx"))
      )
      .drop([
          "row_invalid",
          "frame_invalid",
          "row_idx",
          "first_valid_idx",
          "last_valid_idx",
          "sequence_id",
      ])
)

# Save the cleaned dataset
lf_clean.sink_parquet("../data/landmarks_cleaned.parquet")


#Final sanity check (this should be empty)
check = (
    lf_clean
    .with_columns(
        (
            pl.col("x_norm").is_null() |
            pl.col("y_norm").is_null() |
            pl.col("z_norm").is_null()
        ).alias("row_invalid")
    )
    .group_by(["patient_name", "frame"])
    .agg(
        pl.any("row_invalid").alias("frame_invalid")
    )
    .group_by("patient_name")
    .agg([
        pl.first("frame_invalid").alias("first_frame_invalid"),
        pl.last("frame_invalid").alias("last_frame_invalid"),
    ])
    .collect()
)

print(check.filter(pl.col("first_frame_invalid") | pl.col("last_frame_invalid")))


shape: (12, 3)
┌─────────────────────────────────┬─────────────────────┬────────────────────┐
│ video_id                        ┆ first_frame_invalid ┆ last_frame_invalid │
│ ---                             ┆ ---                 ┆ ---                │
│ str                             ┆ bool                ┆ bool               │
╞═════════════════════════════════╪═════════════════════╪════════════════════╡
│ semantic_segmentation_PA108_FG… ┆ false               ┆ true               │
│ semantic_segmentation_PA203_UG… ┆ false               ┆ true               │
│ semantic_segmentation_PA139_UG… ┆ true                ┆ false              │
│ semantic_segmentation_PA164_FG… ┆ false               ┆ true               │
│ semantic_segmentation_PA026_FG… ┆ false               ┆ true               │
│ …                               ┆ …                   ┆ …                  │
│ semantic_segmentation_PA292_UG… ┆ true                ┆ false              │
│ semantic_segmentation_PA336_FG… ┆ f

## Checks if any frames and rows left

In [83]:
# Confirm no boundary nulls remain (hard assertion)
assert lf_clean.select(
    (
        (pl.col("x_norm").is_not_null() &
         pl.col("y_norm").is_not_null() &
         pl.col("z_norm").is_not_null()).first() &
        (pl.col("x_norm").is_not_null() &
         pl.col("y_norm").is_not_null() &
         pl.col("z_norm").is_not_null()).last()
    )
).collect().item(), "Boundary nulls detected!"


In [39]:
# save the cleaned dataset
lf_clean.sink_parquet("../data/landmarks_cleaned.parquet")

In [84]:
# Spot-check one patient visually
pl.read_parquet("../data/landmarks_cleaned.parquet") \
  .filter(pl.col("patient_name") == "PA002") \
  .select(["frame", "x_norm", "y_norm", "z_norm"]) \
  .head(40)


frame,x_norm,y_norm,z_norm
i64,f64,f64,f64
24,0.076332,0.146762,-0.287472
24,0.07703,0.134778,-0.27491
24,0.077886,0.134157,-0.275155
24,0.078884,0.133728,-0.275114
24,0.074213,0.135684,-0.293483
…,…,…,…
25,0.078985,0.13439,-0.20859
25,0.080293,0.134005,-0.208556
25,0.074149,0.135723,-0.22587


In [85]:
# check cleaned parquet file

pl.scan_parquet("../data/landmarks_cleaned.parquet") \
  .select(pl.len()) \
  .collect()



len
u32
21808888


This should now contain the full frames after removing frames at the beginning and end of the sequence for each video_id that had null values inside the coordinates x_norm, y_norm and z_norm.

## Check for null values for the coordinates and check if they are consecutive

In [55]:


# 1️⃣ Load cleaned dataset
lf_clean = pl.scan_parquet("../data/landmarks_cleaned.parquet")

# 2️⃣ Mark invalid coordinate rows
lf_clean = lf_clean.with_columns(
    (
        pl.col("x_norm").is_null() |
        pl.col("y_norm").is_null() |
        pl.col("z_norm").is_null()
    ).alias("invalid_flag")
)

# 3️⃣ Sort by patient + landmark + frame
lf_clean = lf_clean.sort(["patient_name", "landmark_id", "frame"])

# 4️⃣ Compute frame difference from previous row per patient + landmark
lf_clean = lf_clean.with_columns(
    (pl.col("frame") - pl.col("frame").shift(1).over(["patient_name","landmark_id"])).alias("frame_diff"),
)

# 5️⃣ Identify start of new invalid sequences
lf_clean = lf_clean.with_columns(
    (
        (pl.col("invalid_flag") & 
         (
            (pl.col("frame_diff") != 1) | 
            (~pl.col("invalid_flag").shift(1).over(["patient_name","landmark_id"]).fill_null(False))
         )
        )
        .cast(pl.UInt32)  # 1 = new sequence, 0 = continuation
    ).alias("new_invalid_sequence")
)

# 6️⃣ Assign a unique ID to each consecutive invalid sequence
# cumulative sum of new_invalid_sequence per patient + landmark
lf_clean = lf_clean.with_columns(
    pl.sum("new_invalid_sequence").over(["patient_name","landmark_id"]).alias("invalid_sequence_id")
)

# 7️⃣ Aggregate info per invalid sequence
invalid_sequences = (
    lf_clean
    .filter(pl.col("invalid_flag"))  # only invalid rows
    .group_by(["patient_name","landmark_id","invalid_sequence_id"])
    .agg([
        pl.min("frame").alias("start_frame"),
        pl.max("frame").alias("end_frame"),
        pl.count().alias("num_frames")
    ])
    .sort(["patient_name","landmark_id","start_frame"])
    .collect()
)

print(invalid_sequences)


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/1584208013.py:48: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_frames")


shape: (21, 6)
┌───────────────────────┬─────────────┬─────────────────────┬─────────────┬───────────┬────────────┐
│ patient_name          ┆ landmark_id ┆ invalid_sequence_id ┆ start_frame ┆ end_frame ┆ num_frames │
│ ---                   ┆ ---         ┆ ---                 ┆ ---         ┆ ---       ┆ ---        │
│ str                   ┆ i64         ┆ u32                 ┆ i64         ┆ i64       ┆ u32        │
╞═══════════════════════╪═════════════╪═════════════════════╪═════════════╪═══════════╪════════════╡
│ cljw4bg4q002e3n6lkugx ┆ 0           ┆ 1                   ┆ 900         ┆ 904       ┆ 5          │
│ 2mrd                  ┆             ┆                     ┆             ┆           ┆            │
│ cljw70yj4003d3n6lzhjn ┆ 0           ┆ 1                   ┆ 568         ┆ 579       ┆ 12         │
│ 55nb                  ┆             ┆                     ┆             ┆           ┆            │
│ cljw743d9003q3n6l8tpm ┆ 0           ┆ 1                   ┆ 124         ┆ 

## Adjusted for video_id

In [86]:
# 1️⃣ Load cleaned dataset
lf_clean = pl.scan_parquet("../data/landmarks_cleaned.parquet")

# 2️⃣ Mark invalid coordinate rows
lf_clean = lf_clean.with_columns(
    (
        pl.col("x_norm").is_null() |
        pl.col("y_norm").is_null() |
        pl.col("z_norm").is_null()
    ).alias("invalid_flag")
)

# 3️⃣ Sort by video + landmark + frame
lf_clean = lf_clean.sort(["video_id", "landmark_id", "frame"])

# 4️⃣ Compute frame difference per video + landmark
lf_clean = lf_clean.with_columns(
    (pl.col("frame") - pl.col("frame").shift(1)
        .over(["video_id", "landmark_id"])
    ).alias("frame_diff")
)

# 5️⃣ Identify start of new invalid sequences
lf_clean = lf_clean.with_columns(
    (
        pl.col("invalid_flag") &
        (
            (pl.col("frame_diff") != 1) |
            (
                ~pl.col("invalid_flag")
                  .shift(1)
                  .over(["video_id", "landmark_id"])
                  .fill_null(False)
            )
        )
    )
    .cast(pl.UInt32)
    .alias("new_invalid_sequence")
)

# 6️⃣ Assign a unique ID to each consecutive invalid sequence
lf_clean = lf_clean.with_columns(
    pl.sum("new_invalid_sequence")
      .over(["video_id", "landmark_id"])
      .alias("invalid_sequence_id")
)

# 7️⃣ Aggregate info per invalid sequence
invalid_sequences = (
    lf_clean
    .filter(pl.col("invalid_flag"))
    .group_by(["video_id", "landmark_id", "invalid_sequence_id"])
    .agg([
        pl.min("frame").alias("start_frame"),
        pl.max("frame").alias("end_frame"),
        pl.count().alias("num_frames"),
    ])
    .sort(["video_id", "landmark_id", "start_frame"])
    .collect()
)

print(invalid_sequences)


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2320/4090885916.py:56: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_frames"),


shape: (399, 6)
┌───────────────────────┬─────────────┬─────────────────────┬─────────────┬───────────┬────────────┐
│ video_id              ┆ landmark_id ┆ invalid_sequence_id ┆ start_frame ┆ end_frame ┆ num_frames │
│ ---                   ┆ ---         ┆ ---                 ┆ ---         ┆ ---       ┆ ---        │
│ str                   ┆ i64         ┆ u32                 ┆ i64         ┆ i64       ┆ u32        │
╞═══════════════════════╪═════════════╪═════════════════════╪═════════════╪═══════════╪════════════╡
│ cljw4bg4q002e3n6lkugx ┆ 0           ┆ 1                   ┆ 900         ┆ 904       ┆ 5          │
│ 2mrd_back…            ┆             ┆                     ┆             ┆           ┆            │
│ cljw70yj4003d3n6lzhjn ┆ 0           ┆ 1                   ┆ 568         ┆ 579       ┆ 12         │
│ 55nb_left…            ┆             ┆                     ┆             ┆           ┆            │
│ cljw743d9003q3n6l8tpm ┆ 0           ┆ 1                   ┆ 124         ┆

In [87]:
# get the unique landmarker_ids from invalid sequences
unique_landmarks = invalid_sequences.select("landmark_id").unique()
print(unique_landmarks)


shape: (1, 1)
┌─────────────┐
│ landmark_id │
│ ---         │
│ i64         │
╞═════════════╡
│ 0           │
└─────────────┘


In [89]:
# Count how many patients have at least one invalid sequence
num_patients_with_invalids = (
    invalid_sequences
    .select(pl.col("video_id").unique())
    .height
)

print(f"Number of patients with at least one invalid sequence: {num_patients_with_invalids}")


Number of patients with at least one invalid sequence: 399


In [90]:
# Filter sequences where invalid spans more than 1 frame
long_invalids = invalid_sequences.filter(pl.col("num_frames") > 1)

# Count unique patients affected
num_patients_long_invalids = long_invalids.select(pl.col("video_id").unique()).height

print(f"Number of patients with consecutive null frames > 1: {num_patients_long_invalids}")


Number of patients with consecutive null frames > 1: 182


In [91]:
# If `invalid_sequences` is already an eager DataFrame
# Filter sequences where invalid spans more than 1 frame
long_invalids = invalid_sequences.filter(pl.col("num_frames") > 1)

# Sort for readability
long_invalids = long_invalids.sort(["video_id", "landmark_id", "start_frame"])

# Show the full table directly
print(long_invalids)

shape: (182, 6)
┌───────────────────────┬─────────────┬─────────────────────┬─────────────┬───────────┬────────────┐
│ video_id              ┆ landmark_id ┆ invalid_sequence_id ┆ start_frame ┆ end_frame ┆ num_frames │
│ ---                   ┆ ---         ┆ ---                 ┆ ---         ┆ ---       ┆ ---        │
│ str                   ┆ i64         ┆ u32                 ┆ i64         ┆ i64       ┆ u32        │
╞═══════════════════════╪═════════════╪═════════════════════╪═════════════╪═══════════╪════════════╡
│ cljw4bg4q002e3n6lkugx ┆ 0           ┆ 1                   ┆ 900         ┆ 904       ┆ 5          │
│ 2mrd_back…            ┆             ┆                     ┆             ┆           ┆            │
│ cljw70yj4003d3n6lzhjn ┆ 0           ┆ 1                   ┆ 568         ┆ 579       ┆ 12         │
│ 55nb_left…            ┆             ┆                     ┆             ┆           ┆            │
│ cljxkx17i000c3n6ldi91 ┆ 0           ┆ 1                   ┆ 527         ┆

Export list of patients with null values in their coordinates to a csv file

In [61]:
# Load cleaned dataset (if not already)
lf_clean = pl.read_parquet("../data/landmarks_cleaned.parquet")  # eager DataFrame

# Mark rows with any null coordinates
lf_clean = lf_clean.with_columns(
    (
        pl.col("x_norm").is_null() |
        pl.col("y_norm").is_null() |
        pl.col("z_norm").is_null()
    ).alias("row_invalid")
)

# Select all unique patients who have at least one invalid row
patients_with_invalids = (
    lf_clean
    .filter(pl.col("row_invalid"))
    .select(pl.col("patient_name").unique())
    .sort("patient_name")
)

# Export to CSV
patients_with_invalids.write_csv("../data/patients_with_invalid_coordinates.csv")

print(f"Exported {patients_with_invalids.height} patients to CSV.")















Exported 21 patients to CSV.


In [62]:
# Assuming `invalid_sequences` is your table with consecutive invalid sequences
# Columns: patient_name, landmark_id, invalid_sequence_id, start_frame, end_frame, num_frames

# Filter sequences with at least 1 invalid frame
patients_invalid_full = invalid_sequences.filter(pl.col("num_frames") > 0)

# Sort for readability
patients_invalid_full = patients_invalid_full.sort(["patient_name", "landmark_id", "start_frame"])

# Export to CSV
patients_invalid_full.write_csv("../data/patients_invalid_sequences.csv")

print(f"Exported {patients_invalid_full.height} sequences to CSV.")

Exported 21 sequences to CSV.


check if the missing coordinates are the same or different

In [86]:


# 1️⃣ Read cleaned dataset
lf_clean = pl.read_parquet("../data/landmarks_cleaned.parquet")

# 2️⃣ Keep only patients that had nulls before
patients_with_invalid = long_invalids["patient_name"].to_list()
lf_invalid = lf_clean.filter(pl.col("patient_name").is_in(patients_with_invalid))

# 3️⃣ Mark nulls per coordinate
lf_invalid = lf_invalid.with_columns([
    pl.col("x_norm").is_null().alias("x_null"),
    pl.col("y_norm").is_null().alias("y_null"),
    pl.col("z_norm").is_null().alias("z_null"),
    (pl.col("x_norm").is_null() | pl.col("y_norm").is_null() | pl.col("z_norm").is_null()).alias("row_invalid")
])

# 4️⃣ Assign a row index per patient + landmark to track consecutive frames
lf_invalid = lf_invalid.sort(["patient_name", "landmark_id", "frame"])
lf_invalid = lf_invalid.with_columns(
    pl.int_range(0, pl.count()).over(["patient_name","landmark_id"]).alias("row_idx")
)

# 5️⃣ Detect consecutive invalid sequences per patient + landmark
lf_invalid = lf_invalid.with_columns([
    (
        pl.col("row_invalid") & 
        (~pl.col("row_invalid").shift(1).over(["patient_name","landmark_id"])).fill_null(True)
    ).alias("new_seq_flag")
])

# 6️⃣ Assign sequence IDs by forward filling an integer marker
# We use `pl.when` + `pl.int_range` + forward_fill to create unique IDs without cumsum
new_seq_rows = lf_invalid.filter(pl.col("new_seq_flag")).select([
    "patient_name", "landmark_id", "row_idx"
]).with_columns([
    pl.int_range(1, pl.count() + 1).alias("invalid_sequence_id")
])

lf_invalid = lf_invalid.join(new_seq_rows, on=["patient_name","landmark_id","row_idx"], how="left")
lf_invalid = lf_invalid.with_columns([
    pl.col("invalid_sequence_id").forward_fill().alias("invalid_sequence_id")
])

# 7️⃣ Aggregate per invalid sequence to count frames per coordinate
invalid_coords_summary = (
    lf_invalid
    .filter(pl.col("row_invalid"))
    .group_by(["patient_name","landmark_id","invalid_sequence_id"])
    .agg([
        pl.sum("x_null").alias("num_x_null"),
        pl.sum("y_null").alias("num_y_null"),
        pl.sum("z_null").alias("num_z_null"),
        pl.count("row_invalid").alias("num_frames"),
        pl.min("frame").alias("start_frame"),
        pl.max("frame").alias("end_frame"),
    ])
    .sort(["patient_name","landmark_id","invalid_sequence_id"])
)

# 8️⃣ Export to CSV
invalid_coords_summary.write_csv("../data/coordinates_invalid_sequences.csv")
print(f"Exported {invalid_coords_summary.height} invalid sequences with per-coordinate counts.")


Exported 23 invalid sequences with per-coordinate counts.


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/1791204823.py:19: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.int_range(0, pl.count()).over(["patient_name","landmark_id"]).alias("row_idx")
/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/1791204823.py:35: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.int_range(1, pl.count() + 1).alias("invalid_sequence_id")


## Filter for the joints we want

In [92]:

# 1️⃣ Load the cleaned dataset
lf_clean = pl.read_parquet("../data/landmarks_cleaned.parquet")

# 2️⃣ Define the landmarks you want to keep
GAIT_JOINTS = [
    2, 5,     # eyes (head orientation)
    11, 12,   # shoulders
    23, 24,   # hips
    25, 26,   # knees
    27, 28,   # ankles
    29, 30,   # heels
    31, 32    # foot index
]

# 3️⃣ Filter the dataset to keep only those landmarks
lf_gait = lf_clean.filter(pl.col("landmark_id").is_in(GAIT_JOINTS))

# 4️⃣ Save as a new parquet
lf_gait.write_parquet("../data/landmarks_gait_joints.parquet")

print(f"Saved {lf_gait.height} rows for the selected gait landmarks.")


Saved 9251704 rows for the selected gait landmarks.


In [93]:
# check if there are still null values for the coordinates

# Load the filtered gait joints dataset
lf_gait = pl.read_parquet("../data/landmarks_gait_joints.parquet")

# Check for any nulls per coordinate
null_summary = lf_gait.select([
    pl.col("x_norm").is_null().sum().alias("num_x_null"),
    pl.col("y_norm").is_null().sum().alias("num_y_null"),
    pl.col("z_norm").is_null().sum().alias("num_z_null"),
    pl.count().alias("total_rows")
])

print(null_summary)


shape: (1, 4)
┌────────────┬────────────┬────────────┬────────────┐
│ num_x_null ┆ num_y_null ┆ num_z_null ┆ total_rows │
│ ---        ┆ ---        ┆ ---        ┆ ---        │
│ u32        ┆ u32        ┆ u32        ┆ u32        │
╞════════════╪════════════╪════════════╪════════════╡
│ 0          ┆ 0          ┆ 0          ┆ 9251704    │
└────────────┴────────────┴────────────┴────────────┘


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2320/926496924.py:11: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_rows")


In [94]:
lf_gait.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str,str
1107969,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,5,0.659191,0.317202,-0.080875,0.997372,632.0,171.0,null,null,null,null,null,null,null,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107975,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,11,0.680979,0.388419,-0.135898,0.999967,653.0,209.0,null,null,null,null,null,null,null,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107976,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,12,0.661917,0.387589,0.021804,0.998632,635.0,209.0,null,null,null,null,null,null,null,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107987,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,23,0.662738,0.550898,-0.054501,0.999938,636.0,297.0,null,null,null,null,null,null,null,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107988,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,24,0.667992,0.545544,0.054363,0.999892,641.0,294.0,null,null,null,null,null,null,null,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"


In [95]:
# Get unique values per column
unique_values = {
    "height": lf_gait.select(pl.col("height").unique()).to_series().to_list(),
    "width": lf_gait.select(pl.col("width").unique()).to_series().to_list(),
    "fps": lf_gait.select(pl.col("fps").unique()).to_series().to_list(),
}

print(unique_values)

{'height': [None, 720, 1080], 'width': [None, 1280, 1920], 'fps': [None, 60.0]}


In [96]:
# Fill null values in the columns height, width, fps
# Fill nulls with the given values
lf_filled = lf_gait.with_columns([
    pl.col("height").fill_null(540),
    pl.col("width").fill_null(960),
    pl.col("fps").fill_null(33),
    pl.col("dataset").fill_null('normal'),
    pl.col("gait_pattern").fill_null('normal'),
    pl.col("add_pattern_info").fill_null('normal'),
    pl.col("title").fill_null('normal')
])

In [97]:
lf_filled.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str,str
1107969,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,5,0.659191,0.317202,-0.080875,0.997372,632.0,171.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107975,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,11,0.680979,0.388419,-0.135898,0.999967,653.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107976,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,12,0.661917,0.387589,0.021804,0.998632,635.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107987,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,23,0.662738,0.550898,-0.054501,0.999938,636.0,297.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"
1107988,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,24,0.667992,0.545544,0.054363,0.999892,641.0,294.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…"


In [101]:
#replace Abnormal Gait with abnormal
lf_filled = lf_filled.with_columns(
    pl.col("dataset")
      .str.replace("^Abnormal Gait$", "abnormal")
      .alias("dataset")
)


In [104]:
lf_filled["dataset"].unique()



dataset
str
"""normal"""
"""abnormal"""


In [105]:
#save dataframe
lf_filled.write_parquet("../data/filled_gait_data.parquet")

In [124]:

# Read the Parquet file
df = pl.read_parquet("../data/filled_gait_data.parquet")

# Encode 'dataset': normal -> 0, abnormal -> 1
df = df.with_columns(
    pl.when(pl.col("dataset") == "normal")
      .then(0)
      .otherwise(1)
      .alias("dataset_encoded")
)

# Save back if needed
df.write_parquet("../data/filled_gait_data_encoded.parquet")

print(df)

shape: (9_251_704, 24)
┌──────────┬────────────┬───────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ id       ┆ patient_na ┆ frame ┆ movement_t ┆ … ┆ gait_marke ┆ file_path  ┆ video_id  ┆ dataset_e │
│ ---      ┆ me         ┆ ---   ┆ ype        ┆   ┆ rs         ┆ ---        ┆ ---       ┆ ncoded    │
│ i64      ┆ ---        ┆ i64   ┆ ---        ┆   ┆ ---        ┆ str        ┆ str       ┆ ---       │
│          ┆ str        ┆       ┆ str        ┆   ┆ str        ┆            ┆           ┆ i32       │
╞══════════╪════════════╪═══════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 1107969  ┆ PA027      ┆ 51    ┆ Fast       ┆ … ┆ null       ┆ C:\User_V\ ┆ semantic_ ┆ 0         │
│          ┆            ┆       ┆ Movement   ┆   ┆            ┆ 2_Github\G ┆ segmentat ┆           │
│          ┆            ┆       ┆            ┆   ┆            ┆ AITy-Capst ┆ ion_PA027 ┆           │
│          ┆            ┆       ┆            ┆   ┆            ┆ …   

## Mapping of gait anomaly to marker

In [ ]:
#check the database tables
import sqlite3

# Connect to the database
conn = sqlite3.connect("../data/landmark_database.db")
cursor = conn.cursor()

# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables in the database:")
for t in tables:
    print(t[0])

conn.close()


Tables in the database:
landmarks
sqlite_sequence
processing_log
processing_metrics


In [ ]:
# This is the class definition
LASS_MAP = {
    "gait_anomaly_distal_foot_control_deficit": {
        "Foot Drop", "Foot Slap", "Inadequate Dorsiflexion",
        "Foot Flat Initial Contact", "Excess Pronation", "Excess Supination",
        "Reduced Metatarsophalangeal Joint Extension",
        "Absent Heel Rise During Terminal Stance", "Early Heel Rise", "Steppage Gait",
    },
    "gait_anomaly_knee_sagittal_plane_abnormality": {
        "Knee Extensor Thrust", "Knee Hyperextension", "Reduced Knee Extension",
        "Reduced Knee Flexion", "Knee Valgus",
    },
    "gait_anomaly_hip_pelvic_control_deficit": {
        "Trendelenburg", "Hip Hiking", "Posterior Pelvic Tilt", "Anterior Pelvic Tilt",
        "Reduced Pelvic Rotation", "Reduced Hip Extension", "Reduced Hip Internal Rotation",
        "Circumduction", "Medial Whip",
    },
    "gait_anomaly_trunk_balance_abnormality": {
        "Reduced Arm Swing", "Forward Lean", "Left Lean", "Right Lean",
        "Reduced Trunk Rotation", "Imbalance", "Cautious Gait",
    },
    "gait_anomaly_spatiotemporal_asymmetry": {
        "Wide Base of Support", "Step Length Asymmetry", "Reduced Step Length", "Reduced Left Weightshift",
    },
}

#Mapping doesn't work as in the rows entries are with spaces separated only so create anchors with keywords in each class that appear most often and match to the most appearing keywords
ANCHORS = {
    "gait_anomaly_distal_foot_control_deficit": [
        "Steppage Gait",
        "Absent Heel Rise During Terminal Stance",
        "Early Heel Rise",
    ],
    "gait_anomaly_knee_sagittal_plane_abnormality": [
        "Knee Extensor Thrust",
        "Knee Hyperextension",
        "Knee Valgus",
    ],
    "gait_anomaly_hip_pelvic_control_deficit": [
        "Trendelenburg",
        "Hip Hiking",
        "Circumduction",
        "Anterior Pelvic Tilt",
    ],
    "gait_anomaly_trunk_balance_abnormality": [
        "Reduced Arm Swing",
        "Cautious Gait",
        "Imbalance",
    ],
    "gait_anomaly_spatiotemporal_asymmetry": [
        "Step Length Asymmetry",
        "Reduced Left Weightshift",
        "Wide Base of Support",
    ],
}


In [ ]:
#match to the anchors and assign classes according to that
#this is for assigning the most frequent class only
import polars as pl

# --- Load parquet ---
df = pl.read_parquet("../data/filled_gait_data_encoded.parquet")

# Normalize text
df = df.with_columns(
    pl.col("gait_markers").str.to_lowercase().alias("_txt")
)

# Build anchor-count expressions per class
score_exprs = {}

for class_name, anchors in ANCHORS.items():
    exprs = [
        pl.col("_txt").str.contains(anchor.lower()).cast(pl.Int8)
        for anchor in anchors
    ]
    score_exprs[class_name] = pl.sum_horizontal(exprs).alias(f"{class_name}_score")

# Add score columns
df = df.with_columns(list(score_exprs.values()))

# Assign class with max anchor score
score_cols = list(score_exprs.keys())

df = df.with_columns(
    pl.when(pl.max_horizontal([pl.col(f"{c}_score") for c in score_cols]) > 0)
    .then(
        pl.struct([pl.col(f"{c}_score") for c in score_cols])
        .map_elements(
            lambda s: score_cols[list(s.values()).index(max(s.values()))]
        )
    )
    .otherwise(None)
    .alias("gait_anomaly")
)

# Cleanup
df = df.drop(
    ["_txt"] + [f"{c}_score" for c in score_cols]
)


In [ ]:
#check in gait anoomaly what are the unique values and how many
df.group_by("gait_anomaly").count().sort("count", descending=True)


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_46969/927604596.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df.group_by("gait_anomaly").count().sort("count", descending=True)


gait_anomaly,count
str,u32
null,8438780
"""gait_anomaly_spatiotemporal_as…",306348
"""gait_anomaly_knee_sagittal_pla…",186872
"""gait_anomaly_trunk_balance_abn…",183512
"""gait_anomaly_hip_pelvic_contro…",76916
"""gait_anomaly_distal_foot_contr…",59276


In [10]:
df.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded,gait_anomaly
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str,str,i32,str
1107969,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,5,0.659191,0.317202,-0.080875,0.997372,632.0,171.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,null
1107975,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,11,0.680979,0.388419,-0.135898,0.999967,653.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,null
1107976,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,12,0.661917,0.387589,0.021804,0.998632,635.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,null
1107987,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,23,0.662738,0.550898,-0.054501,0.999938,636.0,297.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,null
1107988,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,24,0.667992,0.545544,0.054363,0.999892,641.0,294.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,null


In [ ]:
#assign multi-label classes based on anchor presence
import polars as pl

# --- Anchors per class ---
ANCHORS = {
    "gait_anomaly_distal_foot_control_deficit": [
        "Steppage Gait",
        "Absent Heel Rise During Terminal Stance",
        "Early Heel Rise",
    ],
    "gait_anomaly_knee_sagittal_plane_abnormality": [
        "Knee Extensor Thrust",
        "Knee Hyperextension",
        "Knee Valgus",
    ],
    "gait_anomaly_hip_pelvic_control_deficit": [
        "Trendelenburg",
        "Hip Hiking",
        "Circumduction",
        "Anterior Pelvic Tilt",
    ],
    "gait_anomaly_trunk_balance_abnormality": [
        "Reduced Arm Swing",
        "Cautious Gait",
        "Imbalance",
    ],
    "gait_anomaly_spatiotemporal_asymmetry": [
        "Step Length Asymmetry",
        "Reduced Left Weightshift",
        "Wide Base of Support",
    ],
}

# --- Load data ---
df = pl.read_parquet("../data/filled_gait_data_encoded.parquet")

df = df.with_columns(
    pl.col("gait_markers").str.to_lowercase().alias("_txt")
)

# --- Compute 0/1 score for each class ---
for class_name, anchors in ANCHORS.items():
    df = df.with_columns(
        pl.sum_horizontal(
            [pl.col("_txt").str.contains(anchor.lower()).cast(pl.Int8) for anchor in anchors]
        ).alias(f"{class_name}_score")
    )

# --- Assign multi-class list using map_elements ---
score_cols = [f"{c}_score" for c in ANCHORS.keys()]

df = df.with_columns(
    pl.struct(score_cols).map_elements(
        lambda s: [
            class_name for class_name, col in zip(ANCHORS.keys(), s.values()) if col > 0
        ]
    ).alias("gait_anomalies")
)

# --- Optional: drop intermediate score columns ---
df = df.drop(["_txt"] + score_cols)

# --- Preview ---
print(df.select(["gait_markers", "gait_anomalies"]).head(10))


shape: (10, 2)
┌──────────────┬────────────────┐
│ gait_markers ┆ gait_anomalies │
│ ---          ┆ ---            │
│ str          ┆ list[str]      │
╞══════════════╪════════════════╡
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
└──────────────┴────────────────┘


In [36]:
import polars as pl

# Explode the gait_anomalies list into multiple rows
df_exploded = df.explode("gait_anomalies")

# Keep only rows where gait_anomalies is not null (i.e., non-empty lists)
df_exploded_nonempty = df_exploded.filter(pl.col("gait_anomalies").is_not_null())

# Count how many times each class appears

class_counts = (
    df_exploded_nonempty
    .group_by("gait_anomalies")
    .count()
    .sort("count", descending=True)
)

print(class_counts)



shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ gait_anomalies                  ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ gait_anomaly_trunk_balance_abn… ┆ 326522 │
│ gait_anomaly_spatiotemporal_as… ┆ 325024 │
│ gait_anomaly_hip_pelvic_contro… ┆ 316652 │
│ gait_anomaly_knee_sagittal_pla… ┆ 291844 │
│ gait_anomaly_distal_foot_contr… ┆ 243096 │
└─────────────────────────────────┴────────┘


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_47489/954653965.py:14: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


In [37]:
#counts how many classes per row are assigned
df_with_counts = df.with_columns(
    pl.col("gait_anomalies").map_elements(lambda lst: len(lst)).alias("num_classes")
)

df_with_counts.group_by("num_classes").count().sort("num_classes")




/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_47489/2877190789.py:6: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df_with_counts.group_by("num_classes").count().sort("num_classes")


num_classes,count
i64,u32
0,8438780
1,340522
2,254590
3,217812


In [38]:
df.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded,gait_anomalies
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str,str,i32,list[str]
1107969,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,5,0.659191,0.317202,-0.080875,0.997372,632.0,171.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[]
1107975,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,11,0.680979,0.388419,-0.135898,0.999967,653.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[]
1107976,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,12,0.661917,0.387589,0.021804,0.998632,635.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[]
1107987,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,23,0.662738,0.550898,-0.054501,0.999938,636.0,297.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[]
1107988,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,24,0.667992,0.545544,0.054363,0.999892,641.0,294.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[]


In [39]:
#add the columns for each class
import polars as pl

df = df.with_columns(
    pl.arange(0, df.height).alias("row_nr")
)

df_exploded = df.explode("gait_anomalies")

df_exploded = df_exploded.with_columns(
    pl.lit(1).alias("present")
)


df_wide = (
    df_exploded
    .pivot(
        values="present",
        index="row_nr",
        columns="gait_anomalies",
        aggregate_function="max"
    )
    .fill_null(0)
)

df = df.join(df_wide, on="row_nr", how="left")



/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_47489/766137137.py:17: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


In [40]:
df.select(df_wide.columns).head()


row_nr,null,gait_anomaly_knee_sagittal_plane_abnormality,gait_anomaly_trunk_balance_abnormality,gait_anomaly_spatiotemporal_asymmetry,gait_anomaly_hip_pelvic_control_deficit,gait_anomaly_distal_foot_control_deficit
i64,i32,i32,i32,i32,i32,i32
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [41]:
df.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded,gait_anomalies,row_nr,null,gait_anomaly_knee_sagittal_plane_abnormality,gait_anomaly_trunk_balance_abnormality,gait_anomaly_spatiotemporal_asymmetry,gait_anomaly_hip_pelvic_control_deficit,gait_anomaly_distal_foot_control_deficit
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str,str,i32,list[str],i64,i32,i32,i32,i32,i32,i32
1107969,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,5,0.659191,0.317202,-0.080875,0.997372,632.0,171.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],0,0,0,0,0,0,0
1107975,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,11,0.680979,0.388419,-0.135898,0.999967,653.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],1,0,0,0,0,0,0
1107976,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,12,0.661917,0.387589,0.021804,0.998632,635.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],2,0,0,0,0,0,0
1107987,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,23,0.662738,0.550898,-0.054501,0.999938,636.0,297.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],3,0,0,0,0,0,0
1107988,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,24,0.667992,0.545544,0.054363,0.999892,641.0,294.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],4,0,0,0,0,0,0


In [42]:
# count 1s
class_cols = list(ANCHORS.keys())

df.select([
    pl.col(c).sum().alias(c) for c in class_cols
]).transpose(include_header=True, header_name="class", column_names=["count"]) \
  .sort("count", descending=True)


class,count
str,i32
"""gait_anomaly_trunk_balance_abn…",326522
"""gait_anomaly_spatiotemporal_as…",325024
"""gait_anomaly_hip_pelvic_contro…",316652
"""gait_anomaly_knee_sagittal_pla…",291844
"""gait_anomaly_distal_foot_contr…",243096


In [43]:
#check unique values in movement type
df.group_by("movement_type").count().sort("count", descending=True)


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_47489/4116507419.py:2: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df.group_by("movement_type").count().sort("count", descending=True)


movement_type,count
str,u32
"""Regular Movement""",4538198
"""Fast Movement""",3069332
"""SLOWMOTION""",866642
"""N""",507822
null,260750
"""until second 6, then slowow""",8960


In [44]:
# Filter rows where gait_anomalies is not null
non_null_df = df.filter(pl.col("gait_anomalies").is_not_null())

# Show first 10 rows with gait_markers and the mapped class
print(non_null_df.select(["gait_markers", "gait_anomalies"]).head(10))

# Optionally, also see how many rows were mapped
print(f"Total rows with gait_anomalies assigned: {non_null_df.height}")

shape: (10, 2)
┌──────────────┬────────────────┐
│ gait_markers ┆ gait_anomalies │
│ ---          ┆ ---            │
│ str          ┆ list[str]      │
╞══════════════╪════════════════╡
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
│ null         ┆ []             │
└──────────────┴────────────────┘
Total rows with gait_anomalies assigned: 9251704


## fill all the gait_anomalies and gait_markers where null and where movement_type = Regular or Fast Movement with normal - not necessary

In [30]:
#fill all the gait_anomalies and gait_markers where null and where movement_type = Regular or Fast Movement with normal
df = df.with_columns([
    # gait_anomalies is List[str]
    pl.when(
        pl.col("movement_type").is_in(["Regular", "Fast Movement"])
        & (pl.col("gait_anomalies").list.len() == 0)
    )
    .then(pl.lit(["normal"]))   # keep column as List[str]
    .otherwise(pl.col("gait_anomalies"))
    .alias("gait_anomalies"),

    # gait_markers is str
    pl.when(
        pl.col("movement_type").is_in(["Regular", "Fast Movement"])
        & pl.col("gait_markers").is_null()
    )
    .then(pl.lit("normal"))
    .otherwise(pl.col("gait_markers"))
    .alias("gait_markers"),
])


In [32]:
# check if in these two columns null values are left
df.select([
    pl.col("gait_anomalies").null_count().alias("gait_anomalies_nulls"),
    pl.col("gait_markers").null_count().alias("gait_markers_nulls"),
])


gait_anomalies_nulls,gait_markers_nulls
u32,u32
0,5332964


In [33]:
# for gait anomalies after filling in the empty entries
df.select(
    (pl.col("gait_anomalies").list.len() == 0).sum().alias("empty_after_fix"),
    pl.col("gait_anomalies").null_count().alias("null_after_fix")
)


empty_after_fix,null_after_fix
u32,u32
5369448,0


In [ ]:
# now fill all the remaining cells in the two columns with abnormal


## continue

In [45]:
df.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded,gait_anomalies,row_nr,null,gait_anomaly_knee_sagittal_plane_abnormality,gait_anomaly_trunk_balance_abnormality,gait_anomaly_spatiotemporal_asymmetry,gait_anomaly_hip_pelvic_control_deficit,gait_anomaly_distal_foot_control_deficit
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str,str,i32,list[str],i64,i32,i32,i32,i32,i32,i32
1107969,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,5,0.659191,0.317202,-0.080875,0.997372,632.0,171.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],0,0,0,0,0,0,0
1107975,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,11,0.680979,0.388419,-0.135898,0.999967,653.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],1,0,0,0,0,0,0
1107976,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,12,0.661917,0.387589,0.021804,0.998632,635.0,209.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],2,0,0,0,0,0,0
1107987,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,23,0.662738,0.550898,-0.054501,0.999938,636.0,297.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],3,0,0,0,0,0,0
1107988,"""PA027""",51,"""Fast Movement""","""Right""",1700.0,24,0.667992,0.545544,0.054363,0.999892,641.0,294.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,"""C:\User_V\2_Github\GAITy-Capst…","""semantic_segmentation_PA027_FG…",0,[],4,0,0,0,0,0,0


In [46]:
# Check for remaining nulls in all columns
print(df.null_count())


shape: (1, 32)
┌─────┬─────────────┬───────┬─────────────┬───┬─────────────┬────────────┬────────────┬────────────┐
│ id  ┆ patient_nam ┆ frame ┆ movement_ty ┆ … ┆ gait_anomal ┆ gait_anoma ┆ gait_anoma ┆ gait_anoma │
│ --- ┆ e           ┆ ---   ┆ pe          ┆   ┆ y_trunk_bal ┆ ly_spatiot ┆ ly_hip_pel ┆ ly_distal_ │
│ u32 ┆ ---         ┆ u32   ┆ ---         ┆   ┆ ance_abn…   ┆ emporal_as ┆ vic_contro ┆ foot_contr │
│     ┆ u32         ┆       ┆ u32         ┆   ┆ ---         ┆ …          ┆ …          ┆ …          │
│     ┆             ┆       ┆             ┆   ┆ u32         ┆ ---        ┆ ---        ┆ ---        │
│     ┆             ┆       ┆             ┆   ┆             ┆ u32        ┆ u32        ┆ u32        │
╞═════╪═════════════╪═══════╪═════════════╪═══╪═════════════╪════════════╪════════════╪════════════╡
│ 0   ┆ 0           ┆ 0     ┆ 260750      ┆ … ┆ 0           ┆ 0          ┆ 0          ┆ 0          │
└─────┴─────────────┴───────┴─────────────┴───┴─────────────┴────────────┴──

In [ ]:
# fill null values for gait markers and gait_anomaly
#lf_filled = lf_filled.with_columns([
 #   pl.col("gait_anomaly").fill_null("normal"),
 #   pl.col("gait_markers").fill_null("normal"),
#   pl.col("movement_type").fill_null("abnormal")
#])

In [47]:
# Save to Parquet
df.write_parquet("../data/clean_gait_data.parquet")



# Steps for data cleaning

1. removed frames with leading or trailing null values for coordinates
2. check of null values for coordinates shows that only landmarker id =0 is missing once in while -> ignore because this is going to be excluded anyways.
3. only kept the landmarkers of interest
4. imputed null values
5. added anomaly classes column and for null values imputed normal
6. added columns for the abnormality classes with 1 and 0

# Selection of columns and encoding

Decided to go for a two model decision approach. 1. Model predicts normal vs abnormal and 2. Model predicts the classes (1-5)

In [49]:
# get the column names of the parquet file
import polars as pl

# Load the parquet file
df = pl.read_parquet("../data/clean_gait_data.parquet")

# Get all column names
columns = df.columns
print(columns)


['id', 'patient_name', 'frame', 'movement_type', 'side', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm', 'z_norm', 'visibility', 'x_px', 'y_px', 'dataset', 'gait_pattern', 'add_pattern_info', 'title', 'fps', 'width', 'height', 'gait_markers', 'file_path', 'video_id', 'dataset_encoded', 'gait_anomalies', 'row_nr', 'null', 'gait_anomaly_knee_sagittal_plane_abnormality', 'gait_anomaly_trunk_balance_abnormality', 'gait_anomaly_spatiotemporal_asymmetry', 'gait_anomaly_hip_pelvic_control_deficit', 'gait_anomaly_distal_foot_control_deficit']


In [ ]:
import polars as pl

PARQUET_PATH = "../data/clean_gait_data.parquet"

COLUMNS_TO_KEEP = [
    "patient_name",
    "frame",
    "landmark_id",
    "x_norm",
    "y_norm",
    "z_norm",
    "fps",
    "dataset",      # binary label source
    "gait_pattern",   # binary label source
    "gait_anomaly",   # multi-class label
]

df = (
    pl.read_parquet(PARQUET_PATH)
      .select(COLUMNS_TO_KEEP)
      .sort(["patient_name", "frame", "landmark_id"])
)

print(df.shape)
print(df.head())


(7908306, 10)
shape: (5, 10)
┌──────────────┬───────┬─────────────┬──────────┬───┬──────┬─────────┬──────────────┬──────────────┐
│ patient_name ┆ frame ┆ landmark_id ┆ x_norm   ┆ … ┆ fps  ┆ dataset ┆ gait_pattern ┆ gait_anomaly │
│ ---          ┆ ---   ┆ ---         ┆ ---      ┆   ┆ ---  ┆ ---     ┆ ---          ┆ ---          │
│ str          ┆ i64   ┆ i64         ┆ f64      ┆   ┆ f64  ┆ str     ┆ str          ┆ str          │
╞══════════════╪═══════╪═════════════╪══════════╪═══╪══════╪═════════╪══════════════╪══════════════╡
│ PA000        ┆ 30    ┆ 2           ┆ 0.910548 ┆ … ┆ 33.0 ┆ normal  ┆ normal       ┆ normal       │
│ PA000        ┆ 30    ┆ 2           ┆ 0.880661 ┆ … ┆ 33.0 ┆ normal  ┆ normal       ┆ normal       │
│ PA000        ┆ 30    ┆ 2           ┆ 0.127009 ┆ … ┆ 33.0 ┆ normal  ┆ normal       ┆ normal       │
│ PA000        ┆ 30    ┆ 2           ┆ 0.162204 ┆ … ┆ 33.0 ┆ normal  ┆ normal       ┆ normal       │
│ PA000        ┆ 30    ┆ 2           ┆ 0.119507 ┆ … ┆ 33.0 ┆ n

In [141]:
# Get unique values of a column, e.g., 'gait_pattern'
unique_labels = df.select(pl.col("gait_anomaly").unique())
print(unique_labels)

shape: (3, 1)
┌─────────────────────────────────┐
│ gait_anomaly                    │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ gait_anomaly_hip_pelvic_contro… │
│ null                            │
│ gait_anomaly_distal_foot_contr… │
└─────────────────────────────────┘


In [6]:
# Get unique values of a column, e.g., 'dataset'
unique_labels = df.select(pl.col("dataset").unique())
print(unique_labels)

shape: (2, 1)
┌───────────────┐
│ dataset       │
│ ---           │
│ str           │
╞═══════════════╡
│ normal        │
│ Abnormal Gait │
└───────────────┘
